# K8 · Full system evaluation (RAG pipeline + Gemini)
Evaluates the **whole ResearchLens pipeline**, not single models:

* **A · QASPER test** (text questions on NLP papers; our evidence finder never saw this split)
  * retrieval: BM25 word overlap vs bge search vs **bge + our SciBERT evidence finder** (Recall@1/3/5, MRR)
  * answers: Gemini **without retrieval** (title + abstract) vs **RAG with bge** vs **full system**
    (QASPER answer F1, evidence F1 of cited passages, citation precision)
* **B · SPIQA test-A** (questions about figures and tables)
  * finding the reference figure: caption search vs caption + re-rank vs **SigLIP image search** vs full system
  * answers: captions only vs **with figure images**, judged correct / incorrect by Gemini against the gold answer

Every number gets a bootstrap 95% confidence interval. Results are saved after every question, so if the session
stops, just run all cells again and it continues. At the end everything is uploaded to the private results
dataset (`results/system_eval/`).

Needs: **GPU T4 x2**, **Internet on**, secrets **`HF_TOKEN`** (write) and **`GEMINI_API_KEY`**.
Time: about 1–1.5 h (the free Gemini tier is paced at one request every 5 s).

## 0. Setup (Kaggle)
Before running: right panel → **Accelerator: GPU T4 x2 (or P100)**, **Internet: On** (needs a phone-verified Kaggle account),
and **Add-ons → Secrets → add `HF_TOKEN`** (your Hugging Face write token) and tick it for this notebook.

Kaggle has no Google Drive: results go to `/kaggle/working/ResearchLens` **and** every model + its results are pushed to your
private Hugging Face account. The last cell also backs up all results to a private HF dataset.

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/mkavinvelavan/ResearchLens.git"
DRIVE_ROOT = "/kaggle/working/ResearchLens"          # results (Kaggle's saved output folder)
SCRATCH = "/tmp/rl"                                    # data, checkpoints, final models (large, not saved)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"               # one GPU, same setup as the Colab runs
for d in (DRIVE_ROOT, f"{SCRATCH}/content"):
    os.makedirs(d, exist_ok=True)
if not os.path.exists("/content"):                     # configs use /content/... paths (Colab layout)
    try:
        os.symlink(f"{SCRATCH}/content", "/content")
    except OSError as e:
        print("note: could not create /content link:", e)
os.makedirs("/content/data", exist_ok=True)

if not os.path.exists(f"{SCRATCH}/repo"):
    !git clone -q {REPO_URL} {SCRATCH}/repo
else:
    !git -C {SCRATCH}/repo pull -q
%cd {SCRATCH}/repo
!pip install -q -r requirements-colab.txt

os.environ.update({
    "RL_DRIVE_ROOT": DRIVE_ROOT, "RL_DATA_DIR": "/content/data",
    "RL_CKPT_DIR": f"{SCRATCH}/checkpoints", "RL_MODEL_DIR": f"{SCRATCH}/final_models",
    "TOKENIZERS_PARALLELISM": "false", "TQDM_DISABLE": "1",
})
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN found")
except Exception as e:
    print("WARNING: HF_TOKEN secret missing / not attached -> models will NOT be pushed to the Hub.", e)
!nvidia-smi --query-gpu=name,memory.total --format=csv

def run(module, config, run_name, seed, extra=""):
    # Train one (run, seed) unless it already finished; streams the log and stops on failure.
    import yaml
    tag = yaml.safe_load(open(config))["model_tag"]
    if os.path.exists(f"{DRIVE_ROOT}/results/{tag}/{run_name}/seed_{seed}/metrics.json"):
        print(f"skip {run_name} seed {seed} (finished)"); return
    if "--push_to_hub" in extra and not os.environ.get("HF_TOKEN"):
        extra = extra.replace("--push_to_hub", "")
    cmd = f"{sys.executable} -m {module} --config {config} --run_name {run_name} --seed {seed} {extra}"
    print(">>>", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line])[-40:]
    if p.wait() != 0:
        raise RuntimeError(f"Training failed: {run_name} seed {seed}. Last lines:\n" + "".join(tail))

## 1. Install the app's packages and load the Gemini key

In [ ]:
!pip install -q -r requirements-app.txt
from kaggle_secrets import UserSecretsClient
os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY"); print("GEMINI_API_KEY loaded")

## 2. Run the evaluation
100 QASPER questions (≈ 300 Gemini calls) and 50 SPIQA questions (≈ 200 calls incl. the judge).
Lower `--n_qasper` / `--n_spiqa` for a quicker run.

In [ ]:
OUT = "/kaggle/working/system_eval"
!python -m researchlens.evaluation.system_eval --out_dir {OUT} --data_dir /kaggle/working/eval_data --n_qasper 100 --n_spiqa 50

## 3. Results

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open(f"{OUT}/report.md").read()))
display(Image(f"{OUT}/system_eval.png"))

## 4. Upload to the private results dataset (`results/system_eval/`)

In [ ]:
from huggingface_hub import HfApi
api = HfApi(); repo = f"{os.environ.get('RL_HF_USER', 'Kavinvelavan')}/researchlens-results"
api.upload_folder(repo_id=repo, repo_type="dataset", folder_path=OUT, path_in_repo="results/system_eval",
                  ignore_patterns=["qdrant/*", "papers/*"], token=os.environ.get("HF_TOKEN"),
                  commit_message="system evaluation")
print("uploaded to", repo, "results/system_eval")